In [ ]:
import requests

url = "https://egov.kz/ru/identity/main"
resp = requests.get(url)
resp.encoding = 'utf-8'  
data = resp.json()
print(data["data"]["title"])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import requests, json

URL = "https://egov.kz/ru/services/presale/S0001"
headers = {
    "Next-Action": "7f6fec875b6e87857069b25607f98651d17fa445a6",
    "Content-Type": "text/plain;charset=UTF-8",
    "Accept": "text/x-component",
}

payload = [{
    "route": {"isNeedAuth": False, "url": "/v1/cms/passports/public/S0001", "method": "GET"},
    "elkSessionId": "",
    "forwardSetCookie": True,
    "queryParams": "$undefined",
    "credentials": "$undefined",
}]

resp = requests.post(URL, headers=headers, data=json.dumps(payload), timeout=30)
print(resp.status_code)
print(resp.content.decode("utf-8")[:800])

200
0:{"a":"$@1","f":"","b":"pC2r2yDOfqiL9DBsFtPWc"}
2:Tb39,<section style="margin:0 0 24px">
    <h2 style="margin:0 0 12px;font-size:20px;font-weight:700;line-height:1.35">Как получить услугу в госоргане или организации услугодателя (необходимые документы)</h2>

    <ul style="margin:0 0 16px;padding-left:22px;list-style:disc">
        <li style="margin:0 0 6px;line-height:1.55">Для подачи заявки по Абайской, Западно-Казахстанской, Туркестанской, Кызылординской, Павлодарской, Восточно-Казахстанской области, а также по отдельным районам других областей: Костанайская область (Аулиекольский район, Денисовский район, Житикаринский район, Камыстинский район, Лисаковск Г.А., Сарыкольский район, Узункольский район), Северо-Казахстанская область (Айыртауский район, Акжарский район, Жамбылский район,


In [ ]:
text = resp.content.decode("utf-8")
print(len(text))
import re

print(re.findall(r'(?m)^([0-9a-f]+):', text))

for line in text.split("\n"):
    if line.startswith("1:"):
        print(line[:1500])

9546
['0', '2']


In [ ]:
URL_KK = "https://egov.kz/kk/services/presale/S0001"
resp_kk = requests.post(URL_KK, headers=headers, data=json.dumps(payload), timeout=30)
print(resp_kk.status_code)
print(resp_kk.content.decode("utf-8")[:800])

200
0:{"a":"$@1","f":"","b":"pC2r2yDOfqiL9DBsFtPWc"}
2:Tb39,<section style="margin:0 0 24px">
    <h2 style="margin:0 0 12px;font-size:20px;font-weight:700;line-height:1.35">Как получить услугу в госоргане или организации услугодателя (необходимые документы)</h2>

    <ul style="margin:0 0 16px;padding-left:22px;list-style:disc">
        <li style="margin:0 0 6px;line-height:1.55">Для подачи заявки по Абайской, Западно-Казахстанской, Туркестанской, Кызылординской, Павлодарской, Восточно-Казахстанской области, а также по отдельным районам других областей: Костанайская область (Аулиекольский район, Денисовский район, Житикаринский район, Камыстинский район, Лисаковск Г.А., Сарыкольский район, Узункольский район), Северо-Казахстанская область (Айыртауский район, Акжарский район, Жамбылский район,


In [ ]:
text = resp.content.decode("utf-8")


print(text[-1200:])


for key in ['"title"', '"how_it_works"', '"kk"', '"ru"', '"en"', '"cost"', '"waiting_time"']:
    print(key, text.count(key))


from bs4 import BeautifulSoup  
plain = BeautifulSoup(text.split("2:", 1)[1], "html.parser").get_text(" ", strip=True)
print(len(plain))
print(plain[:600])

ащения к услугодателю, в Государственную корпорацию, на портале – 30 минут","kk":"Көрсетілетін қызметті берушіге, Мемлекеттік корпорацияға, порталға жүгінген сәтінен бастап – 30 минут","en":"From the moment of contacting the service provider, the State Corporation, on the portal – 30 minutes"},"cost":{"ru":"Бесплатно","kk":"Тегін","en":"Free of charge"},"provider":{"ru":"Министерство просвещения РК","kk":"ҚР Оқу-ағарту министрлігі","en":"Ministry of Education of the Republic of Kazakhstan"},"recipients":{"ru":"Физические лица","kk":"Жеке тұлғалар","en":"Individuals"},"delivery_methods":[],"additionally":[{"link_url":"https://zan.gov.kz/client/#!/doc/143994/rus/282","text":{"ru":"Правила оказания государственной услуги","kk":"Мемлекеттік қызмет көрсету ережесі","en":"Rules for providing State service"}}],"how_to_get_service_at_psc":{"ru":"$2","kk":"$3","en":"$4"}},"is_automated":true,"order_url":"https://egov.kz/services/S0001/","requires_auth":true,"require_auth_for_fl":true,"require_a

In [ ]:
import json
from bs4 import BeautifulSoup

def parse_flight(b: bytes) -> dict:
    rows, i = {}, 0
    while i < len(b):
        colon = b.index(b":", i)
        rid = b[i:colon].decode().strip()
        j = colon + 1
        if b[j:j+1] == b"T":                      
            comma = b.index(b",", j)
            n = int(b[j+1:comma], 16)             
            start = comma + 1
            rows[rid] = b[start:start+n].decode("utf-8")
            i = start + n
        else:                                    
            nl = b.find(b"\n", j)
            nl = len(b) if nl == -1 else nl
            rows[rid] = b[j:nl].decode("utf-8")
            i = nl + 1
    return rows

def html_to_text(html):
    return BeautifulSoup(html, "html.parser").get_text(" ", strip=True)

rows = parse_flight(resp.content)
print(list(rows.keys()))          

record = json.loads(rows["1"])["data"]
psc = record["details"]["how_to_get_service_at_psc"]
for lang in ("ru", "kk", "en"):
    ref = psc[lang]                              
    if isinstance(ref, str) and ref.startswith("$"):
        psc[lang] = html_to_text(rows[ref[1:]])

print(record["title"])
print(record["details"]["cost"])
print(psc["kk"][:300])

['0', '2', '3', '4', '1']
{'ru': 'Постановка на очередь детей дошкольного возраста (до 6 лет)', 'kk': 'Мектепке дейінгі ұйымдарға жіберу үшін мектеп жасына дейінгі балаларды (6 жасқа дейін) кезекке қою', 'en': 'Putting children (up to 6 years old) into a waiting list to kindergarten'}
{'ru': 'Бесплатно', 'kk': 'Тегін', 'en': 'Free of charge'}
Мемлекеттік органда немесе қызмет көрсетуші ұйымда қызметті қалай алуға болады (қажетті құжаттар) Абай, Батыс Қазақстан, Түркістан, Қызылорда, Павлодар, Шығыс Қазақстан облыстары бойынша, сондай-ақ басқа облыстардың жекелеген аудандары бойынша өтінім беру үшін: Қостанай облысы (Әулиекөл ауданы, Дени


In [ ]:
import time, json, requests

ACTION_ID = "7f6fec875b6e87857069b25607f98651d17fa445a6"

def fetch_service(code):
    url = f"https://egov.kz/ru/services/presale/{code}"
    headers = {
        "Next-Action": ACTION_ID,
        "Content-Type": "text/plain;charset=UTF-8",
        "Accept": "text/x-component",
    }
    payload = [{
        "route": {"isNeedAuth": False,
                  "url": f"/v1/cms/passports/public/{code}",
                  "method": "GET"},
        "elkSessionId": "",
        "forwardSetCookie": True,
        "queryParams": "$undefined",
        "credentials": "$undefined",
    }]

    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=30)
    if r.status_code != 200:
        print(code, "-> HTTP", r.status_code)
        return None

    rows = parse_flight(r.content)
    if "1" not in rows:
        print(code, "-> no row 1")
        return None

    obj = json.loads(rows["1"])
    if "data" not in obj:
        print(code, "-> no data:", str(obj)[:150])
        return None

    rec = obj["data"]
    psc = rec["details"]["how_to_get_service_at_psc"]
    for lang in ("ru", "kk", "en"):
        ref = psc.get(lang)
        if isinstance(ref, str) and ref.startswith("$"):
            psc[lang] = html_to_text(rows[ref[1:]])
    return rec

for code in ["S0001", "P719", "P1001", "S0002"]:
    rec = fetch_service(code)
    print(code, "->", rec["title"]["kk"][:60] if rec else "no result")
    time.sleep(1)

S0001 -> Мектепке дейінгі ұйымдарға жіберу үшін мектеп жасына дейінгі
P719 -> Ауруханаға жатқызылған жағдайда хабарлама жіберу үшін сенімд
P1001 -> Соттылықтың болуы не болмауы туралы анықтама (электрондық)
S0002 -> no data: {'code': 'PASSPORT_ERROR_NOT_FOUND', 'type': 'ERROR', 'message': {'kk': 'Қызмет паспорты табылмады', 'ru': 'Паспорт услуги не найден', 'en': 'Service 
S0002 -> no result


In [ ]:
import re, json, time, requests

sm = requests.get("https://egov.kz/sitemap.xml", timeout=30).text
codes = sorted(set(re.findall(r"/services/([A-Za-z0-9]+)/steps/", sm)))
print(len(codes), "codes from sitemap")

records = {}
for code in codes:
    rec = fetch_service(code)
    if rec:
        records[code] = rec
    time.sleep(1)

print(f"{len(records)} / {len(codes)} fetched")

with open("egov_records.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)


empty_kk = [c for c, r in records.items() if not r["title"]["kk"].strip()]
same_kk_ru = [c for c, r in records.items() if r["title"]["kk"] == r["title"]["ru"]]
print("empty Kazakh titles:", empty_kk)
print("Kazakh title identical to Russian:", same_kk_ru)

31 codes from sitemap
BMG -> no data: {'code': 'PASSPORT_ERROR_NOT_FOUND', 'type': 'ERROR', 'message': {'kk': 'Қызмет паспорты табылмады', 'ru': 'Паспорт услуги не найден', 'en': 'Service 
P2203 -> no data: {'code': 'PASSPORT_ERROR_NOT_FOUND', 'type': 'ERROR', 'message': {'kk': 'Қызмет паспорты табылмады', 'ru': 'Паспорт услуги не найден', 'en': 'Service 
29 / 31 fetched
empty Kazakh titles: []
Kazakh title identical to Russian: []


In [ ]:
import numpy as np

def body_text(rec, lang):
    parts = [rec["title"][lang]]
    parts += [s["text"][lang] for s in rec.get("how_it_works", [])]
    parts.append(rec["details"]["how_to_get_service_at_psc"][lang])
    return " ".join(p for p in parts if p)

stats = []
for c, r in records.items():
    kk, ru = len(body_text(r, "kk")), len(body_text(r, "ru"))
    stats.append((c, kk, ru, kk / ru if ru else 0))

ratios = [s[3] for s in stats]
print("median kk/ru length ratio:", round(float(np.median(ratios)), 2))
for c, kk, ru, ratio in sorted(stats, key=lambda s: s[3])[:5]:
    print(c, "kk:", kk, "ru:", ru, "ratio:", round(ratio, 2))

median kk/ru length ratio: 1.01
P1005 kk: 788 ru: 827 ratio: 0.95
P6504 kk: 448 ru: 467 ratio: 0.96
P4006 kk: 9969 ru: 10343 ratio: 0.96
P641 kk: 4429 ru: 4566 ratio: 0.97
P657 kk: 5634 ru: 5789 ratio: 0.97


In [ ]:
import json, requests, time

ACTION_ID = "7f6fec875b6e87857069b25607f98651d17fa445a6"

def parse_flight(b: bytes) -> dict:
    rows, i = {}, 0
    while i < len(b):
        colon = b.index(b":", i)
        rid = b[i:colon].decode().strip()
        j = colon + 1
        if b[j:j+1] == b"T":
            comma = b.index(b",", j)
            n = int(b[j+1:comma], 16)
            start = comma + 1
            rows[rid] = b[start:start+n].decode("utf-8")
            i = start + n
        else:
            nl = b.find(b"\n", j)
            nl = len(b) if nl == -1 else nl
            rows[rid] = b[j:nl].decode("utf-8")
            i = nl + 1
    return rows

def call_backend(page_path, route_url, query_params="$undefined"):
    url = f"https://egov.kz{page_path}"
    headers = {
        "Next-Action": ACTION_ID,
        "Content-Type": "text/plain;charset=UTF-8",
        "Accept": "text/x-component",
    }
    payload = [{
        "route": {"url": route_url, "isNeedAuth": False, "method": "GET"},
        "elkSessionId": "",
        "forwardSetCookie": True,
        "queryParams": query_params,
        "credentials": "$undefined",
    }]
    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=30)
    if r.status_code != 200:
        print("HTTP", r.status_code)
        return None
    return parse_flight(r.content)

In [ ]:
CAT = "250d6e08-6ad1-4072-bda2-dd223ddd2ce3"
SUB = "83de537f-cc78-49d8-a170-c135ec893014"

def list_services(category_id, subcategory_id=None, page="1", size="100"):
    qp = {"categoryId": category_id, "page": page, "size": size}
    if subcategory_id:
        qp["subcategoryId"] = subcategory_id
    rows = call_backend("/ru/navigation/family", "/v1/cms/passports/by-category", qp)
    if not rows or "1" not in rows:
        return None
    obj = json.loads(rows["1"])
    return obj.get("data")

a = list_services(CAT, SUB)
print("with subcategory:", a["total"] if a else None,
      [i["service_code"] for i in a["items"]] if a else "")


b = list_services(CAT)
print("category only:", b["total"] if b else None,
      [i["service_code"] for i in b["items"]] if b else "")

with subcategory: 6 ['S0001', 'Com013', 'P204', 'P215', 'P6132', 'P657']
category only: 6 ['S0001', 'Com013', 'P204', 'P215', 'P6132', 'P657']


In [ ]:

rows = call_backend("/ru/navigation/family", "/v1/cms/dictionaries/category-tree")
tree = json.loads(rows["1"])
print(json.dumps(tree, ensure_ascii=False)[:2500])


rows = call_backend("/ru/navigation/family", "/v1/cms/passports/popular", {"limit": "100"})
pop = json.loads(rows["1"])
items = pop.get("data", {}).get("items", pop.get("data", []))
print(len(items), "popular services")
print([i.get("service_code") for i in items][:20])

{"data": [{"id": "2115760a-ccb5-4523-9c2c-82900df0ea28", "system_name": "fl", "name": {"ru": "Гражданам", "kk": "Азаматтарға", "en": "For Citizens"}, "children": [{"id": "250d6e08-6ad1-4072-bda2-dd223ddd2ce3", "system_name": "family", "name": {"ru": "Семья", "kk": "Отбасы", "en": "Family"}, "children": [{"id": "83de537f-cc78-49d8-a170-c135ec893014", "system_name": "motherhood", "name": {"ru": "Родительство", "kk": "Ата-ана болу", "en": "Parenting"}, "children": []}, {"id": "31550c59-3886-4bbe-a9c6-5b16e36298ec", "system_name": "children", "name": {"ru": "Ребенок", "kk": "Бала", "en": "Child"}, "children": []}, {"id": "2532c56f-c11d-44b4-893f-8c069abe328b", "system_name": "status", "name": {"ru": "Семейное положение", "kk": "Отбасылық жағдай", "en": "Marital Status"}, "children": []}, {"id": "c322ca75-6de3-40ff-a4a0-d6acafa7eabb", "system_name": "help", "name": {"ru": "Опека, попечительство, поддержка", "kk": "Қамқоршылық, қорғаншылық, қолдау", "en": "Guardianship, Custody, Support"}, "

In [ ]:
def list_services(category_id, subcategory_id=None, page="1", size="100"):
    qp = {"categoryId": category_id, "page": page, "size": size}
    if subcategory_id:
        qp["subcategoryId"] = subcategory_id
    rows = call_backend("/ru/navigation/family", "/v1/cms/passports/by-category", qp)
    if not rows or "1" not in rows:
        return None
    return json.loads(rows["1"]).get("data")

def walk(nodes, path=()):
    for n in nodes:
        p = path + (n,)
        if n.get("children"):
            yield from walk(n["children"], p)
        else:
            yield p

top = tree["data"]
print("top-level:", [n["name"]["en"] for n in top])

leaves = [p for p in walk(top) if len(p) >= 2]
print(len(leaves), "category/subcategory leaves")

services = {}      
sum_totals = 0
for path in leaves:
    cat = path[1]["id"]
    sub = path[2]["id"] if len(path) >= 3 else None
    label = " > ".join(n["name"]["en"] for n in path)
    page, got = 1, 0
    while True:
        data = list_services(cat, sub, page=str(page))
        if not data:
            print("  failed:", label)
            break
        for it in data["items"]:
            services.setdefault(it["service_code"], {
                "title": it["title"],
                "breadcrumb": it["category_breadcrumb"],
            })
        got += len(data["items"])
        if got >= data["total"] or not data["items"]:
            sum_totals += data["total"]
            break
        page += 1
        time.sleep(1)
    print(f"{label}: {got}")
    time.sleep(1)

print(len(services), "unique service codes;", sum_totals, "listings before dedupe")
popular = ['P3001','P3041','P1001','RemoteBMG','P704','P8001','P601','REG02','P305','RemoteEDS','P3061']
print("popular codes missing from crawl:", [c for c in popular if c not in services])

with open("service_codes.json", "w", encoding="utf-8") as f:
    json.dump(services, f, ensure_ascii=False, indent=1)

top-level: ['For Citizens', 'For Business']
88 category/subcategory leaves
For Citizens > Family > Parenting: 6
For Citizens > Family > Child: 17
For Citizens > Family > Marital Status: 19
For Citizens > Family > Guardianship, Custody, Support: 15
For Citizens > Healthcare > Medical Care: 21
For Citizens > Healthcare > Pharmaceutics: 3
For Citizens > Healthcare > Certification and Attestation: 0
For Citizens > Education > Preschool Education: 6
For Citizens > Education > Secondary Education: 18
For Citizens > Education > Colleges: 12
For Citizens > Education > Higher Education: 10
For Citizens > Education > Postgraduate Education: 19
For Citizens > Employment > Job Search and Employment: 10
For Citizens > Employment > Labor Relations: 2
For Citizens > Employment > Civil Service: 6
For Citizens > Employment > Professional Sports: 9
For Citizens > Social Security > Pension Provision: 14
For Citizens > Social Security > Support for Persons with Disabilities: 19
For Citizens > Social Secur

In [ ]:
import os

OUT = "egov_full.jsonl"
done = set()
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as f:
        for line in f:
            done.add(json.loads(line)["service_code"])

todo = [c for c in sorted(services) if c not in done]
print(len(todo), "to fetch;", len(done), "already done")

failed, consecutive = [], 0
with open(OUT, "a", encoding="utf-8") as f:
    for n, code in enumerate(todo, 1):
        try:
            rec = fetch_service(code)
        except Exception as e:
            print(code, "error:", e)
            rec = None
        if rec:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            consecutive = 0
        else:
            failed.append(code)
            consecutive += 1
            if consecutive >= 10:
                print("10 failures in a row, stopping (the Next-Action ID may have changed)")
                break
        if n % 50 == 0:
            print(n, "/", len(todo))
        time.sleep(1)

print("failed:", len(failed), failed[:20])

0 to fetch; 939 already done
failed: 0 []


In [ ]:
import json, numpy as np

recs = []
with open("egov_full.jsonl", encoding="utf-8") as f:
    for line in f:
        recs.append(json.loads(line))
print(len(recs), "records")

def body_text(rec, lang):
    parts = [rec["title"].get(lang, "")]
    parts += [s["text"].get(lang, "") for s in rec.get("how_it_works", [])]
    parts.append(rec["details"]["how_to_get_service_at_psc"].get(lang, "") or "")
    return " ".join(p for p in parts if p)

empty_kk = [r["service_code"] for r in recs if not r["title"].get("kk", "").strip()]
same_title = [r["service_code"] for r in recs if r["title"].get("kk") == r["title"].get("ru")]
print("empty Kazakh titles:", len(empty_kk))
print("Kazakh title identical to Russian:", len(same_title))

stats = []
for r in recs:
    kk, ru = len(body_text(r, "kk")), len(body_text(r, "ru"))
    stats.append((r["service_code"], kk, ru, kk / ru if ru else 0))

ratios = np.array([s[3] for s in stats])
print("median kk/ru:", round(float(np.median(ratios)), 2),
      "| 5th-95th percentile:", np.round(np.percentile(ratios, [5, 95]), 2))
print("services with ratio < 0.7:", int((ratios < 0.7).sum()),
      "| > 1.4:", int((ratios > 1.4).sum()))
print("lowest 5:", [(c, kk, ru, round(x, 2)) for c, kk, ru, x in sorted(stats, key=lambda s: s[3])[:5]])

short = [s[0] for s in stats if s[2] < 200]
print("services with very short Russian body (<200 chars):", len(short))

939 records
empty Kazakh titles: 0
Kazakh title identical to Russian: 0
median kk/ru: 0.99 | 5th-95th percentile: [0.93 1.14]
services with ratio < 0.7: 3 | > 1.4: 5
lowest 5: [('approval_documents_pass131_mtk', 5800, 22212, 0.26), ('state_support_measures_financingofexport', 6513, 16849, 0.39), ('secondary_school_pass_mp_203', 981, 1411, 0.7), ('state_support_measures_investmentprojects', 11017, 15393, 0.72), ('university_degree_204pass_mon', 706, 917, 0.77)]
services with very short Russian body (<200 chars): 33


In [ ]:
by_code = {r["service_code"]: r for r in recs}

def section_report(code):
    r = by_code[code]
    print("==", code)
    for lang in ("ru", "kk"):
        steps = r.get("how_it_works", [])
        psc = r["details"]["how_to_get_service_at_psc"].get(lang, "") or ""
        print(f"  {lang}: title={len(r['title'].get(lang,''))} "
              f"steps={len(steps)} steps_chars={sum(len(s['text'].get(lang,'')) for s in steps)} "
              f"psc_chars={len(psc)}")

for code in ["approval_documents_pass131_mtk", "state_support_measures_financingofexport"]:
    section_report(code)


long_kk = [s[0] for s in stats if s[3] > 1.4]
print("Kazakh >> Russian:", long_kk)

== approval_documents_pass131_mtk
  ru: title=42 steps=5 steps_chars=673 psc_chars=21491
  kk: title=36 steps=5 steps_chars=670 psc_chars=5088
== state_support_measures_financingofexport
  ru: title=66 steps=2 steps_chars=360 psc_chars=16420
  kk: title=74 steps=2 steps_chars=362 psc_chars=6074
Kazakh >> Russian: ['Com04', 'SR14', 'buy_sale_pass2701002_mps', 'customs_2fpass351_mf', 'military_service_101012-1pass_mo']


In [ ]:
r = by_code["approval_documents_pass131_mtk"]
psc = r["details"]["how_to_get_service_at_psc"]
for lang in ("ru", "kk"):
    t = psc[lang]
    print(f"--- {lang}: {len(t)} chars, {t.count('. ')} sentences ---")
    print("START:", t[:300])
    print("END:  ", t[-300:])

--- ru: 21491 chars, 24 sentences ---
START: Как получить услугу в НАО «Правительство для граждан» (необходимые документы) Для получения свидетельства члена летного экипажа заявитель предоставляет следующие документы: в отношении свидетельств частного пилота PPL (самолета, вертолета, дирижабля, воздушного судна с системой увеличения подъемной 
END:   ) или цифровое фото (3,5*4,5 см., не менее 413*531 (пикселей)) при замене на свидетельство в виде пластиковой карточки; электронная копия образца личной подписи при замене на свидетельство в виде пластиковой карточки; электронная копия сведений, подтверждающих оплату оказания государственной услуги.
--- kk: 5088 chars, 3 sentences ---
START: «Азаматтарға арналған үкімет» мемлекеттік корпорациясында қызметті қалай алуға болады (қажетті құжаттар) Ұшу экипажы мүшесінің куәлігін алу үшін өтініш беруші мынадай құжаттарды ұсынады: жеке пилот PPL (ұшақ, тікұшақ, дирижабль, көтергіш күшті арттыру жүйесі бар әуе кемесі), коммерциялық авиация пил
EN

In [ ]:
for code in ["Com04", "SR14"]:
    section_report(code)

for code in ["Com04", "SR14"]:
    r = by_code[code]
    psc = r["details"]["how_to_get_service_at_psc"]
    for lang in ("ru", "kk"):
        t = psc[lang] or ""
        print(f"--- {code} / {lang}: {len(t)} chars ---")
        print("START:", t[:300])
        print("END:  ", t[-300:])

== Com04
  ru: title=72 steps=3 steps_chars=279 psc_chars=0
  kk: title=265 steps=3 steps_chars=275 psc_chars=0
== SR14
  ru: title=55 steps=3 steps_chars=124 psc_chars=0
  kk: title=161 steps=3 steps_chars=110 psc_chars=0
--- Com04 / ru: 0 chars ---
START: 
END:   
--- Com04 / kk: 0 chars ---
START: 
END:   
--- SR14 / ru: 0 chars ---
START: 
END:   
--- SR14 / kk: 0 chars ---
START: 
END:   


In [ ]:
import sys
sys.path.append("../src")
from chunking import chunk_text          # adjust the import name if yours differs

EXCLUDE = {"Com04", "SR14", "approval_documents_pass131_mtk", "state_support_measures_financingofexport"}

def build_corpus(records, lang, chunk_size=200):
    chunks, queries = [], []
    for r in records:
        code = r["service_code"]
        if code in EXCLUDE:
            continue
        title = (r["title"].get(lang) or "").strip()
        steps = " ".join(s["text"].get(lang, "") for s in r.get("how_it_works", []))
        psc = r["details"]["how_to_get_service_at_psc"].get(lang, "") or ""
        body = " ".join(p for p in [steps, psc] if p).strip()
        if not title or not body:
            continue
        for piece in chunk_text(body, chunk_size=chunk_size):   
            chunks.append({"text": piece, "doc_id": code})
        queries.append({"query": title, "relevant_doc": code})
    return chunks, queries

kk_chunks, kk_queries = build_corpus(recs, "kk")
ru_chunks, ru_queries = build_corpus(recs, "ru")

print(f"Kazakh: {len(kk_chunks)} chunks, {len(kk_queries)} queries")
print(f"Russian: {len(ru_chunks)} chunks, {len(ru_queries)} queries")

Kazakh: 1393 chunks, 919 queries
Russian: 1424 chunks, 919 queries


In [ ]:
from evaluate import evaluate_retriever
from retrievers import RETRIEVER_REGISTRY

for lang, chunks, queries in [("kk", kk_chunks, kk_queries), ("ru", ru_chunks, ru_queries)]:
    print(f"=== {lang} ===")
    for name, cls in RETRIEVER_REGISTRY.items():
        retriever = cls(chunks)
        results = evaluate_retriever(retriever, queries, top_k=5)
        print(name, results)

=== kk ===
tfidf {'precision@k': np.float64(0.06354733405875952), 'recall@k': np.float64(0.21871599564744287), 'mrr': np.float64(0.15783460282916212)}
bm25 {'precision@k': np.float64(0.06507072905331882), 'recall@k': np.float64(0.22415669205658323), 'mrr': np.float64(0.1675371780921291)}
hybrid {'precision@k': np.float64(0.06637649619151251), 'recall@k': np.float64(0.22633297062023938), 'mrr': np.float64(0.172506347479144)}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dense {'precision@k': np.float64(0.003264417845484222), 'recall@k': np.float64(0.01632208922742111), 'mrr': np.float64(0.008233587232499093)}
=== ru ===
tfidf {'precision@k': np.float64(0.06224156692056584), 'recall@k': np.float64(0.2295973884657236), 'mrr': np.float64(0.16334784185709106)}
bm25 {'precision@k': np.float64(0.05941240478781283), 'recall@k': np.float64(0.20783460282916214), 'mrr': np.float64(0.1584149437794704)}
hybrid {'precision@k': np.float64(0.06376496191512514), 'recall@k': np.float64(0.22742110990206746), 'mrr': np.float64(0.1689517591585056)}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dense {'precision@k': np.float64(0.003482045701849837), 'recall@k': np.float64(0.017410228509249184), 'mrr': np.float64(0.008995284729778744)}


In [ ]:
from sentence_transformers import SentenceTransformer
if 'e5_model' not in dir():
    e5_model = SentenceTransformer("intfloat/multilingual-e5-large")

from retrievers import DenseRetriever

for lang, chunks, queries in [("kk", kk_chunks, kk_queries), ("ru", ru_chunks, ru_queries)]:
    t0 = time.time()
    retriever = DenseRetriever(chunks, model=e5_model)
    results = evaluate_retriever(retriever, queries, top_k=5)
    print(f"{lang} dense (E5): {results}  ({time.time()-t0:.1f}s)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

In [ ]:

short_services = set()
for r in recs:
    for lang in ("ru", "kk"):
        steps = " ".join(s["text"].get(lang, "") for s in r.get("how_it_works", []))
        psc = r["details"]["how_to_get_service_at_psc"].get(lang, "") or ""
        body = (steps + " " + psc).strip()
        if len(body) < 200:
            short_services.add(r["service_code"])
            break

print(len(short_services), "services with short body in at least one language")

EXCLUDE_FULL = EXCLUDE | short_services
print(len(EXCLUDE_FULL), "total excluded")

def build_corpus_v2(records, lang, chunk_size=200):
    chunks, queries = [], []
    for r in records:
        code = r["service_code"]
        if code in EXCLUDE_FULL:
            continue
        title = (r["title"].get(lang) or "").strip()
        steps = " ".join(s["text"].get(lang, "") for s in r.get("how_it_works", []))
        psc = r["details"]["how_to_get_service_at_psc"].get(lang, "") or ""
        body = " ".join(p for p in [steps, psc] if p).strip()
        if not title or not body:
            continue
        for piece in chunk_text(body, chunk_size=chunk_size):
            chunks.append({"text": piece, "doc_id": code})
        queries.append({"query": title, "relevant_doc": code})
    return chunks, queries



kk_chunks2, kk_queries2 = build_corpus_v2(recs, "kk")
ru_chunks2, ru_queries2 = build_corpus_v2(recs, "ru")
print(f"Kazakh: {len(kk_chunks2)} chunks, {len(kk_queries2)} queries")
print(f"Russian: {len(ru_chunks2)} chunks, {len(ru_queries2)} queries")

for lang, chunks, queries in [("kk", kk_chunks2, kk_queries2), ("ru", ru_chunks2, ru_queries2)]:
    print(f"=== {lang} (short-filtered) ===")
    for name, cls in RETRIEVER_REGISTRY.items():
        if name == "dense":
            continue
        retriever = cls(chunks)
        results = evaluate_retriever(retriever, queries, top_k=5)
        print(name, results)

for lang, chunks, queries in [("kk", kk_chunks2, kk_queries2), ("ru", ru_chunks2, ru_queries2)]:
    retriever = DenseRetriever(chunks, model=e5_model)
    results = evaluate_retriever(retriever, queries, top_k=5)
    print(f"{lang} dense (E5, short-filtered): {results}")

62 services with short body in at least one language
65 total excluded
Kazakh: 1348 chunks, 874 queries
Russian: 1379 chunks, 874 queries
=== kk (short-filtered) ===
tfidf {'precision@k': np.float64(0.06613272311212814), 'recall@k': np.float64(0.22654462242562928), 'mrr': np.float64(0.16487414187643024)}
bm25 {'precision@k': np.float64(0.06796338672768877), 'recall@k': np.float64(0.2334096109839817), 'mrr': np.float64(0.1757246376811594)}
hybrid {'precision@k': np.float64(0.06910755148741418), 'recall@k': np.float64(0.2334096109839817), 'mrr': np.float64(0.17930968726163232)}
=== ru (short-filtered) ===
tfidf {'precision@k': np.float64(0.06475972540045767), 'recall@k': np.float64(0.23684210526315788), 'mrr': np.float64(0.16880244088482071)}
bm25 {'precision@k': np.float64(0.06132723112128146), 'recall@k': np.float64(0.2128146453089245), 'mrr': np.float64(0.16191838291380628)}
hybrid {'precision@k': np.float64(0.06567505720823799), 'recall@k': np.float64(0.2322654462242563), 'mrr': np.f

In [ ]:
kazakh_mrr = 0.158
russian_mrr = 0.222

relative_gap = (russian_mrr - kazakh_mrr) / kazakh_mrr
print(f"Relative gap: {relative_gap:.1%}")

Relative gap: 40.5%
